# Final-label day-level model re-evaluation

This notebook re-scores existing saved model predictions against the current final labelled Alpha/Beta datasets. It is intentionally fast: it does **not** retrain models, does **not** run new model inference, and writes CSV outputs only.

## Inputs

- Final truth: `dataset/final/dataset_alpha.parquet` and `dataset/final/dataset_beta.parquet`
- Beta review confidence: archived reviewer-B annotations from `dataset/oracle_data_creation/archive/2026-07-02_reviewer_B_final/reviewer_B.csv`
- Existing prediction artifacts from Notebook 2 and previous misc experiments

## Outputs

CSV files are written to `notebooks/99_Misc/outputs/10_final_label_day_model_reevaluation/`. The main file is `01_model_day_metrics.csv`, with pooled, macro-site-average, and per-site day-level precision/recall/F1.

## Important interpretation note

Prediction files may have stale label columns from earlier oracle versions. This notebook ignores those embedded labels and always joins predictions to the current final labels from `dataset/final`.


## Run the re-evaluation

This cell builds the model registry, joins each stored prediction artifact to the new final labels, computes day-level P/R/F1, and writes CSV outputs. It does not import model libraries or run model training/prediction.


In [ ]:

from __future__ import annotations

import json
from pathlib import Path
from typing import Iterable

import pandas as pd

RUN_REEVALUATION = True
OUTPUT_FOLDER_NAME = "10_final_label_day_model_reevaluation"


def find_repo_root() -> Path:
    """Find the PyNRPF repo root from either the repo root or a notebook cwd."""
    start = Path.cwd().resolve()
    candidates = [start, *start.parents]
    marker = Path("publication/2_journal_article/dataset/final/dataset_alpha.parquet")
    for candidate in candidates:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing publication/2_journal_article/dataset/final/dataset_alpha.parquet")


ROOT = find_repo_root()
JOURNAL = ROOT / "publication/2_journal_article"
MISC_OUTPUTS = JOURNAL / "notebooks/99_Misc/outputs"
OUT = MISC_OUTPUTS / OUTPUT_FOLDER_NAME
OUT.mkdir(parents=True, exist_ok=True)

# Keep this folder CSV-only on reruns.
for existing in OUT.iterdir():
    if existing.is_file():
        existing.unlink()

FINAL_DATASET_DIR = JOURNAL / "dataset/final"
ORACLE_ARCHIVE = JOURNAL / "dataset/oracle_data_creation/archive/2026-07-02_reviewer_B_final"
NOTEBOOK2_PRED_DIR = JOURNAL / "outputs/intermediate/02_correction_validation"

TRUE_STRINGS = {"true", "t", "1", "yes", "y"}


def date_key(values: pd.Series) -> pd.Series:
    """Return stable YYYY-MM-DD date strings."""
    return pd.to_datetime(values, errors="coerce").dt.strftime("%Y-%m-%d")


def as_bool(values: pd.Series) -> pd.Series:
    """Parse booleans robustly from bool, numeric, or string CSV columns."""
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(values):
        return values.fillna(0).astype(float).ne(0)
    return values.fillna(False).astype(str).str.strip().str.lower().isin(TRUE_STRINGS)


def safe_div(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator else 0.0


def prf_from_counts(tp: int, fp: int, fn: int) -> tuple[float, float, float]:
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = safe_div(2 * precision * recall, precision + recall)
    return precision, recall, f1


def load_final_truth(dataset: str) -> pd.DataFrame:
    """Load final day-level truth, using final label_day from dataset/final."""
    path = FINAL_DATASET_DIR / f"dataset_{dataset}.parquet"
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_parquet(path, columns=["substation_id", "date", "label_day"])
    df["date"] = date_key(df["date"])
    df["label_day"] = as_bool(df["label_day"])
    truth = (
        df.groupby(["substation_id", "date"], as_index=False)["label_day"]
        .max()
        .rename(columns={"label_day": "true_day"})
    )
    truth["dataset"] = dataset
    return truth


def load_beta_confidence() -> pd.DataFrame:
    """Load archived reviewer-B confidence and map raw act_* ids to final beta_* ids."""
    path = ORACLE_ARCHIVE / "reviewer_B.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    conf = pd.read_csv(path)
    required = {"substation_id", "date", "confidence"}
    missing = required.difference(conf.columns)
    if missing:
        raise ValueError(f"Reviewer-B confidence file missing columns: {sorted(missing)}")
    conf = conf[["substation_id", "date", "confidence"]].copy()
    conf["substation_id"] = conf["substation_id"].astype(str).str.replace("^act_", "beta_", regex=True)
    conf["date"] = date_key(conf["date"])
    conf["confidence"] = conf["confidence"].fillna("unknown").astype(str).str.strip().str.lower()
    return conf.drop_duplicates(["substation_id", "date"], keep="last")


alpha_truth = load_final_truth("alpha")
beta_truth = load_final_truth("beta")
beta_confidence = load_beta_confidence()
beta_truth = beta_truth.merge(beta_confidence, on=["substation_id", "date"], how="left")
beta_truth["confidence"] = beta_truth["confidence"].fillna("missing")

truth_by_dataset = {
    "alpha": alpha_truth,
    "beta": beta_truth,
}
truth_source_by_dataset = {
    "alpha": str((FINAL_DATASET_DIR / "dataset_alpha.parquet").relative_to(ROOT)),
    "beta": str((FINAL_DATASET_DIR / "dataset_beta.parquet").relative_to(ROOT)),
}

prediction_sets: list[dict] = []
manifest_rows: list[dict] = []


def add_skip(model_family: str, model_variant: str, dataset: str, source: str, reason: str, notes: str = "") -> None:
    manifest_rows.append({
        "model_family": model_family,
        "model_variant": model_variant,
        "dataset": dataset,
        "status": "skipped",
        "reason": reason,
        "prediction_source": source,
        "source_rows": 0,
        "prediction_days": 0,
        "duplicate_days_collapsed": 0,
        "extra_prediction_days_without_truth": 0,
        "evaluated_days": 0,
        "notes": notes,
    })


def register_predictions(
    model_family: str,
    model_variant: str,
    dataset: str,
    pred_df: pd.DataFrame,
    source: str,
    notes: str = "",
) -> None:
    """Register one model/dataset prediction table after normalising and auditing keys."""
    required = {"substation_id", "date", "pred_day"}
    missing = required.difference(pred_df.columns)
    if missing:
        add_skip(model_family, model_variant, dataset, source, f"missing required prediction columns: {sorted(missing)}", notes)
        return

    pred = pred_df[["substation_id", "date", "pred_day"]].copy()
    pred["substation_id"] = pred["substation_id"].astype(str)
    pred["date"] = date_key(pred["date"])
    pred["pred_day"] = as_bool(pred["pred_day"])
    pred = pred.dropna(subset=["date"])

    source_rows = len(pred)
    duplicate_days = int(pred.duplicated(["substation_id", "date"]).sum())
    pred = pred.groupby(["substation_id", "date"], as_index=False)["pred_day"].max()

    truth = truth_by_dataset[dataset][["substation_id", "date"]]
    truth_keys = set(map(tuple, truth.to_numpy()))
    pred_keys = set(map(tuple, pred[["substation_id", "date"]].to_numpy()))
    extra_days = len(pred_keys.difference(truth_keys))
    evaluated_days = len(pred_keys.intersection(truth_keys))

    status = "included" if evaluated_days else "skipped"
    reason = "ok" if evaluated_days else "no prediction days overlap final truth"
    manifest_rows.append({
        "model_family": model_family,
        "model_variant": model_variant,
        "dataset": dataset,
        "status": status,
        "reason": reason,
        "prediction_source": source,
        "source_rows": source_rows,
        "prediction_days": len(pred),
        "duplicate_days_collapsed": duplicate_days,
        "extra_prediction_days_without_truth": extra_days,
        "evaluated_days": evaluated_days,
        "notes": notes,
    })
    if evaluated_days:
        prediction_sets.append({
            "model_family": model_family,
            "model_variant": model_variant,
            "dataset": dataset,
            "predictions": pred,
            "prediction_source": source,
            "notes": notes,
        })


def load_interval_predictions(paths: Iterable[Path]) -> pd.DataFrame:
    """Collapse interval prediction CSVs to one day-level prediction per site-day."""
    frames = []
    for path in paths:
        df = pd.read_csv(path, usecols=["substation_id", "date", "pred_interval"])
        df["date"] = date_key(df["date"])
        df["pred_interval"] = as_bool(df["pred_interval"])
        frames.append(df)
    if not frames:
        return pd.DataFrame(columns=["substation_id", "date", "pred_day"])
    combined = pd.concat(frames, ignore_index=True)
    return (
        combined.groupby(["substation_id", "date"], as_index=False)["pred_interval"]
        .max()
        .rename(columns={"pred_interval": "pred_day"})
    )


def load_decoded_days(path: Path, pred_col: str) -> pd.DataFrame:
    df = pd.read_csv(path, usecols=["substation_id", "date", pred_col])
    return df.rename(columns={pred_col: "pred_day"})


def decode_scored_candidates(
    scored_path: Path,
    universe_path: Path,
    threshold: float,
    filters: list[tuple[str, str, float]] | None = None,
) -> pd.DataFrame:
    """Decode saved candidate scores to day predictions over a saved day universe."""
    universe = pd.read_csv(universe_path, usecols=["substation_id", "date"])
    universe["date"] = date_key(universe["date"])
    universe = universe.drop_duplicates(["substation_id", "date"])
    universe["pred_day"] = False

    scored = pd.read_csv(scored_path)
    if scored.empty:
        return universe
    scored["date"] = date_key(scored["date"])
    passed = scored["candidate_probability"].astype(float).ge(float(threshold))
    for col, op, value in filters or []:
        if col not in scored.columns or pd.isna(value):
            continue
        series = scored[col].astype(float)
        if op == ">=":
            passed &= series.ge(float(value))
        elif op == "<=":
            passed &= series.le(float(value))
        else:
            raise ValueError(f"Unsupported filter operator: {op}")
    decoded = (
        scored.assign(pred_day=passed)
        .groupby(["substation_id", "date"], as_index=False)["pred_day"]
        .max()
    )
    return universe.drop(columns=["pred_day"]).merge(decoded, on=["substation_id", "date"], how="left").assign(
        pred_day=lambda d: d["pred_day"].where(d["pred_day"].notna(), False).astype(bool)
    )


def read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


# ---------------------------------------------------------------------------
# Register m7/m8 Notebook 2 predictions.
# ---------------------------------------------------------------------------
for method, family in [("m7_dtr", "m7"), ("m8_xgb", "m8")]:
    alpha_files = sorted(NOTEBOOK2_PRED_DIR.glob(f"*_correction_predictions_alpha_holdout_alpha_*_{method}.csv"))
    if alpha_files:
        register_predictions(
            family,
            method,
            "alpha",
            load_interval_predictions(alpha_files),
            ";".join(str(p.relative_to(ROOT)) for p in alpha_files),
            "Collapsed day prediction from any pred_interval=True across saved Alpha holdout predictions.",
        )
    else:
        add_skip(family, method, "alpha", str(NOTEBOOK2_PRED_DIR.relative_to(ROOT)), "missing Alpha holdout prediction files")

    beta_file = NOTEBOOK2_PRED_DIR / f"22_correction_predictions_beta_transfer_{method}.csv"
    if method == "m7_dtr":
        beta_file = NOTEBOOK2_PRED_DIR / "23_correction_predictions_beta_transfer_m7_dtr.csv"
    if beta_file.exists():
        register_predictions(
            family,
            method,
            "beta",
            load_interval_predictions([beta_file]),
            str(beta_file.relative_to(ROOT)),
            "Collapsed day prediction from any pred_interval=True across saved Beta transfer predictions.",
        )
    else:
        add_skip(family, method, "beta", str(beta_file.relative_to(ROOT)), "missing Beta transfer prediction file")

# ---------------------------------------------------------------------------
# Register m9 family exploratory predictions.
# ---------------------------------------------------------------------------
m9_base = MISC_OUTPUTS / "03_m9_hybrid_development"
m9_manifest = read_json(m9_base / "manifests/run_manifest.json")
m9_threshold = float(m9_manifest.get("threshold", 0.15))
alpha_scored = m9_base / "intermediate/04_alpha_loso_scored_candidates.csv"
alpha_universe = m9_base / "intermediate/03_alpha_candidate_day_summary.csv"
if alpha_scored.exists() and alpha_universe.exists():
    register_predictions(
        "m9_hybrid",
        f"base_threshold_{m9_threshold:g}",
        "alpha",
        decode_scored_candidates(alpha_scored, alpha_universe, m9_threshold),
        str(alpha_scored.relative_to(ROOT)),
        "Decoded from saved Alpha scored candidates using saved threshold; no model rerun.",
    )
else:
    add_skip("m9_hybrid", "base", "alpha", str(m9_base.relative_to(ROOT)), "missing Alpha scored candidates or day universe")

m9_beta_decoded = m9_base / "intermediate/07_beta_decoded_days.csv"
if m9_beta_decoded.exists():
    register_predictions(
        "m9_hybrid",
        f"base_threshold_{m9_threshold:g}",
        "beta",
        load_decoded_days(m9_beta_decoded, "pred_day"),
        str(m9_beta_decoded.relative_to(ROOT)),
        "Read saved Beta decoded days.",
    )
else:
    add_skip("m9_hybrid", "base", "beta", str(m9_beta_decoded.relative_to(ROOT)), "missing Beta decoded-day file")

v1b_base = MISC_OUTPUTS / "04_m9_v1b_precision_gate_search"
selected_gate_path = v1b_base / "metrics/01_selected_gate.csv"
if alpha_scored.exists() and alpha_universe.exists() and selected_gate_path.exists():
    gate = pd.read_csv(selected_gate_path).iloc[0]
    filters = [
        ("solar_p95_inside", ">=", gate.get("min_solar_p95", 0.0)),
        ("derivative_same_sign_fraction", ">=", gate.get("min_same_sign", 0.0)),
        ("solar_net_corr", ">=", gate.get("min_corr", -1.0)),
        ("solar_bell_score", ">=", gate.get("min_solar_bell", 0.0)),
        ("net_load_n_shape_score", ">=", gate.get("min_net_shape", 0.0)),
        ("duration_hours", "<=", gate.get("max_duration_hours", 8.0)),
    ]
    register_predictions(
        "m9_hybrid",
        "v1b_precision_gate",
        "alpha",
        decode_scored_candidates(alpha_scored, alpha_universe, float(gate["threshold"]), filters),
        str(alpha_scored.relative_to(ROOT)),
        f"Decoded from saved Alpha scored candidates using selected gate {gate['gate_id']}.",
    )
else:
    add_skip("m9_hybrid", "v1b_precision_gate", "alpha", str(v1b_base.relative_to(ROOT)), "missing gate file or Alpha scored candidates")

v1b_beta_decoded = v1b_base / "intermediate/02_beta_selected_gate_decoded_days.csv"
if v1b_beta_decoded.exists():
    register_predictions(
        "m9_hybrid",
        "v1b_precision_gate",
        "beta",
        load_decoded_days(v1b_beta_decoded, "pred_day"),
        str(v1b_beta_decoded.relative_to(ROOT)),
        "Read saved Beta selected-gate decoded days.",
    )
else:
    add_skip("m9_hybrid", "v1b_precision_gate", "beta", str(v1b_beta_decoded.relative_to(ROOT)), "missing Beta selected-gate decoded-day file")

pseudo_base = MISC_OUTPUTS / "05_m9_pseudoload_iterative_search"
pseudo_pred = pseudo_base / "intermediate/06_best_variant_predictions.csv"
pseudo_variant = "best_variant"
leaderboard = pseudo_base / "metrics/02_variant_leaderboard.csv"
if leaderboard.exists():
    pseudo_variant = str(pd.read_csv(leaderboard, nrows=1).iloc[0]["variant_id"])
if pseudo_pred.exists():
    register_predictions(
        "m9_pseudoload",
        pseudo_variant,
        "beta",
        load_decoded_days(pseudo_pred, "pred_day"),
        str(pseudo_pred.relative_to(ROOT)),
        "Best pseudo-load exploratory variant; Beta-guided and not publication validation.",
    )
else:
    add_skip("m9_pseudoload", pseudo_variant, "beta", str(pseudo_pred.relative_to(ROOT)), "missing best-variant prediction file")
add_skip("m9_pseudoload", pseudo_variant, "alpha", str(pseudo_base.relative_to(ROOT)), "no row-level Alpha predictions saved for this exploratory run")

# ---------------------------------------------------------------------------
# Register m9.2 predictions.
# ---------------------------------------------------------------------------
physics_base = MISC_OUTPUTS / "07_m9_2_physics_counterfactual_ranker/csv"
for dataset, rel in [("alpha", "full_loso_03_alpha_decoded_days.csv"), ("beta", "full_loso_07_beta_decoded_days.csv")]:
    path = physics_base / rel
    if path.exists():
        register_predictions(
            "m9.2_physics",
            "full_loso_ranker",
            dataset,
            load_decoded_days(path, "pred_label_day"),
            str(path.relative_to(ROOT)),
            "Read saved full-LOSO decoded day predictions.",
        )
    else:
        add_skip("m9.2_physics", "full_loso_ranker", dataset, str(path.relative_to(ROOT)), "missing decoded-day file")

bridge_base = MISC_OUTPUTS / "08_m9_2_bridge_score_development/csv"
bridge_variants = [
    ("v2_alpha_strict", "pred_day_v2_alpha_strict"),
    ("v2_dev_best", "pred_day_v2_dev_best"),
]
for dataset, rel in [("alpha", "01_alpha_daily_bridge_scores.csv"), ("beta", "01_beta_daily_bridge_scores.csv")]:
    path = bridge_base / rel
    if not path.exists():
        for variant, _ in bridge_variants:
            add_skip("m9.2_bridge", variant, dataset, str(path.relative_to(ROOT)), "missing daily bridge score file")
        continue
    df = pd.read_csv(path)
    for variant, pred_col in bridge_variants:
        if pred_col in df.columns:
            register_predictions(
                "m9.2_bridge",
                variant,
                dataset,
                df[["substation_id", "date", pred_col]].rename(columns={pred_col: "pred_day"}),
                str(path.relative_to(ROOT)),
                "Read saved deterministic bridge-score daily prediction column.",
            )
        else:
            add_skip("m9.2_bridge", variant, dataset, str(path.relative_to(ROOT)), f"missing prediction column {pred_col}")

balanced_base = MISC_OUTPUTS / "09_m9_2_bridge_balanced_search"
if balanced_base.exists():
    add_skip("m9.2_bridge_balanced", "search_outputs", "alpha", str(balanced_base.relative_to(ROOT)), "aggregate/search metrics only; no row-level day predictions found")
    add_skip("m9.2_bridge_balanced", "search_outputs", "beta", str(balanced_base.relative_to(ROOT)), "aggregate/search metrics only; no row-level day predictions found")

cgpt_base = MISC_OUTPUTS / "99_cgpt_pro_experiments"
if cgpt_base.exists():
    add_skip("m10_cgpt_pro", "reported_best", "alpha", str(cgpt_base.relative_to(ROOT)), "provided artifacts contain metrics/sweeps, not final-label-safe row-level day predictions")
    add_skip("m10_cgpt_pro", "reported_best", "beta", str(cgpt_base.relative_to(ROOT)), "provided artifacts contain metrics/sweeps, not final-label-safe row-level day predictions")


def metric_counts(df: pd.DataFrame) -> dict:
    true = as_bool(df["true_day"])
    pred = as_bool(df["pred_day"])
    tp = int((true & pred).sum())
    fp = int((~true & pred).sum())
    fn = int((true & ~pred).sum())
    tn = int((~true & ~pred).sum())
    precision, recall, f1 = prf_from_counts(tp, fp, fn)
    return {
        "support": int(len(df)),
        "positive_support": int(true.sum()),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def make_metric_row(base: dict, subset: str, summary_scope: str, substation_id: str, counts: dict, notes: str) -> dict:
    row = {
        "model_family": base["model_family"],
        "model_variant": base["model_variant"],
        "dataset": base["dataset"],
        "subset": subset,
        "summary_scope": summary_scope,
        "substation_id": substation_id,
        **counts,
        "prediction_source": base["prediction_source"],
        "truth_source": truth_source_by_dataset[base["dataset"]],
        "notes": notes,
    }
    return row


metric_rows: list[dict] = []
audit_frames: list[pd.DataFrame] = []

for item in prediction_sets:
    dataset = item["dataset"]
    truth = truth_by_dataset[dataset]
    joined = item["predictions"].merge(truth, on=["substation_id", "date"], how="inner")
    joined["pred_day"] = as_bool(joined["pred_day"])
    joined["true_day"] = as_bool(joined["true_day"])
    if dataset == "alpha":
        joined["confidence"] = "not_applicable"
        subsets = [("all_alpha", joined)]
    else:
        subsets = [
            ("all_beta", joined),
            ("beta_sure_only", joined[joined["confidence"].eq("sure")].copy()),
        ]

    audit = joined[["substation_id", "date", "true_day", "pred_day", "confidence"]].copy()
    audit.insert(0, "dataset", dataset)
    audit.insert(0, "model_variant", item["model_variant"])
    audit.insert(0, "model_family", item["model_family"])
    audit["is_beta_sure"] = audit["confidence"].eq("sure") if dataset == "beta" else False
    audit["prediction_source"] = item["prediction_source"]
    audit_frames.append(audit)

    for subset, sub in subsets:
        if sub.empty:
            continue
        pooled_counts = metric_counts(sub)
        metric_rows.append(make_metric_row(item, subset, "pooled", "", pooled_counts, item["notes"]))

        site_metric_rows = []
        for site, site_df in sub.groupby("substation_id", sort=True):
            counts = metric_counts(site_df)
            site_metric_rows.append(counts)
            metric_rows.append(make_metric_row(item, subset, "site", site, counts, item["notes"]))

        site_metrics_df = pd.DataFrame(site_metric_rows)
        macro_counts = {
            "support": int(site_metrics_df["support"].sum()),
            "positive_support": int(site_metrics_df["positive_support"].sum()),
            "tp": int(site_metrics_df["tp"].sum()),
            "fp": int(site_metrics_df["fp"].sum()),
            "fn": int(site_metrics_df["fn"].sum()),
            "tn": int(site_metrics_df["tn"].sum()),
            "precision": float(site_metrics_df["precision"].mean()),
            "recall": float(site_metrics_df["recall"].mean()),
            "f1": float(site_metrics_df["f1"].mean()),
        }
        metric_rows.append(
            make_metric_row(
                item,
                subset,
                "macro_site_average",
                "",
                macro_counts,
                item["notes"] + " Macro metrics are unweighted means of site-level metrics; counts are summed for context.",
            )
        )

metrics = pd.DataFrame(metric_rows)
metrics = metrics[
    [
        "model_family",
        "model_variant",
        "dataset",
        "subset",
        "summary_scope",
        "substation_id",
        "support",
        "positive_support",
        "tp",
        "fp",
        "fn",
        "tn",
        "precision",
        "recall",
        "f1",
        "prediction_source",
        "truth_source",
        "notes",
    ]
].sort_values(["dataset", "subset", "model_family", "model_variant", "summary_scope", "substation_id"])

manifest = pd.DataFrame(manifest_rows).sort_values(["dataset", "model_family", "model_variant", "status"])

confidence_summary = (
    beta_truth.groupby(["substation_id", "confidence"], as_index=False)
    .agg(site_days=("date", "count"), rpf_days=("true_day", "sum"))
    .sort_values(["substation_id", "confidence"])
)
confidence_totals = (
    beta_truth.groupby(["confidence"], as_index=False)
    .agg(site_days=("date", "count"), rpf_days=("true_day", "sum"))
)
confidence_totals.insert(0, "substation_id", "ALL")
confidence_summary = pd.concat([confidence_totals, confidence_summary], ignore_index=True)

audit = pd.concat(audit_frames, ignore_index=True) if audit_frames else pd.DataFrame()
audit = audit.sort_values(["dataset", "model_family", "model_variant", "substation_id", "date"])

metrics.to_csv(OUT / "01_model_day_metrics.csv", index=False)
manifest.to_csv(OUT / "02_model_source_manifest.csv", index=False)
confidence_summary.to_csv(OUT / "03_beta_confidence_filter_summary.csv", index=False)
audit.to_csv(OUT / "04_joined_day_predictions_audit.csv", index=False)

# Acceptance checks. These deliberately inspect outputs without model training.
assert len(beta_truth) == 2928, f"Expected 2928 Beta site-days, found {len(beta_truth)}"
assert beta_truth["confidence"].eq("sure").any(), "No sure-confidence Beta rows found"
assert not metrics.empty, "No metrics were generated"
assert set(p.suffix for p in OUT.iterdir() if p.is_file()) == {".csv"}, "Output folder contains non-CSV files"
assert audit.duplicated(["model_family", "model_variant", "dataset", "substation_id", "date"]).sum() == 0, "Duplicate joined prediction audit keys found"

print(f"Wrote {len(metrics):,} metric rows for {len(prediction_sets)} included model/dataset prediction sets.")
print(f"Output folder: {OUT.relative_to(ROOT)}")
print("Included prediction sets:")
print(manifest[manifest["status"].eq("included")][["dataset", "model_family", "model_variant", "evaluated_days"]].to_string(index=False))
